# Orbit propagation demo: modeling + validation + visualization

Этот ноутбук показывает полный демонстрационный pipeline проекта:
1. Установка зависимостей (локально и в Google Colab).
2. Задание начальных условий и характеристик спутника.
3. Прогноз орбиты через модуль `dynamics`.
4. Сравнение прогноза с «референсом» на базе реального TLE (CelesTrak + SGP4).
5. Визуализация в 3D (с полупрозрачной Землей) и 2D ground track карте.


## Какие пакеты и методы используются
- **`dynamics.propagator`**: `PropagationConfig`, `propagate_orbit`.
- **`visualization.orbit_3d`**: 3D траектории, полупрозрачная Земля, несколько орбит.
- **`visualization.map_2d`**: 2D ground tracks на карте Земли.
- **`sgp4` + данные CelesTrak**: референсная траектория реального спутника (ISS/Starlink/CubeSat).
- **`astropy`**: конвертация TEME -> ITRS (ECEF) для ground track.

> В текущей реализации `dynamics` использует fallback-модель центрального поля Земли.
> Архитектурно предусмотрено подключение TudatPy backend для high-fidelity сил (J2, drag, Sun/Moon, SRP).


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/<your-org>/<your-repo>.git'  # TODO: set your real repository URL
REPO_DIR = Path('/content/space_modeling') if IN_COLAB else Path.cwd()


def run(cmd):
    print('>>', ' '.join(cmd))
    subprocess.check_call(cmd)

if IN_COLAB:
    if not REPO_DIR.exists():
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-e', '.'])
print('Dependencies installed. Working dir:', Path.cwd())


In [ ]:
import urllib.request
from dataclasses import asdict

import numpy as np
from sgp4.api import Satrec
from sgp4.conveniences import sat_epoch_datetime

from dynamics.propagator import PropagationConfig, SpacecraftProperties, propagate_orbit
from visualization.orbit_3d import build_orbit_figure
from visualization.map_2d import build_groundtrack_figure

from astropy.time import Time
from astropy import units as u
from astropy.coordinates import TEME, ITRS, CartesianRepresentation


## Начальные условия и параметры спутника

Задаем массово-баллистические характеристики (масса, Cd, Cr, площадь) и initial state.


In [ ]:
spacecraft = SpacecraftProperties(
    mass=420000.0,      # кг (пример для ISS)
    cd=2.2,
    cr=1.3,
    reference_area=400.0,
)

initial_state = np.array([
    6778e3, 0.0, 0.0,
    0.0, 7.67e3, 0.0,
])

config = PropagationConfig(
    initial_state=initial_state,
    epoch_seconds=0.0,
    duration_seconds=6 * 5400.0,  # несколько витков
    step_seconds=60.0,
    integrator='DOP853',
    spacecraft=spacecraft,
)
print(asdict(spacecraft))


## Прогноз орбиты (модель проекта)


In [ ]:
t_model, states_model = propagate_orbit(config)
states_model[:3], states_model.shape


## Реальные данные: берем TLE с CelesTrak и строим референс через SGP4


In [ ]:
tle_url = 'https://celestrak.org/NORAD/elements/stations.txt'
raw = urllib.request.urlopen(tle_url, timeout=30).read().decode('utf-8')
lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]

# Берем ISS (обычно первая тройка в stations.txt)
name, l1, l2 = lines[0], lines[1], lines[2]
print('Satellite:', name)
print(l1)
print(l2)

sat = Satrec.twoline2rv(l1, l2)
epoch_dt = sat_epoch_datetime(sat)
print('TLE epoch:', epoch_dt)


In [ ]:
# Синхронизируем горизонт времени с model propagation
minutes = t_model / 60.0
jd0, fr0 = sat.jdsatepoch, sat.jdsatepochF

states_sgp4_teme = np.zeros((len(minutes), 6))
for i, m in enumerate(minutes):
    jd = jd0
    fr = fr0 + m / (24.0 * 60.0)
    e, r_km, v_km_s = sat.sgp4(jd, fr)
    if e != 0:
        raise RuntimeError(f'SGP4 error code {e} at step {i}')
    states_sgp4_teme[i, :3] = np.array(r_km) * 1e3
    states_sgp4_teme[i, 3:] = np.array(v_km_s) * 1e3

states_sgp4_teme.shape


## Сравнение траекторий (модель vs референс SGP4)


In [ ]:
position_error_m = np.linalg.norm(states_model[:, :3] - states_sgp4_teme[:, :3], axis=1)
velocity_error_m_s = np.linalg.norm(states_model[:, 3:] - states_sgp4_teme[:, 3:], axis=1)

print('Position error [m]: min/mean/max =', position_error_m.min(), position_error_m.mean(), position_error_m.max())
print('Velocity error [m/s]: min/mean/max =', velocity_error_m_s.min(), velocity_error_m_s.mean(), velocity_error_m_s.max())


## 3D визуализация (несколько траекторий + полупрозрачная Земля)


In [ ]:
fig3d = build_orbit_figure(
    trajectories=[states_model, states_sgp4_teme],
    names=['Model (two-body fallback)', f'Reference SGP4: {name}'],
    show_earth=True,
    earth_opacity=0.35,
)
fig3d.show()


## 2D визуализация ground tracks на карте Земли


In [ ]:
# TEME -> ITRS(ECEF) для корректной карты
obstime = Time(epoch_dt) + (t_model * u.s)

def teme_to_itrs_xyz(states_teme):
    rep = CartesianRepresentation(states_teme[:, 0] * u.m, states_teme[:, 1] * u.m, states_teme[:, 2] * u.m)
    teme = TEME(rep, obstime=obstime)
    itrs = teme.transform_to(ITRS(obstime=obstime))
    xyz = np.vstack([itrs.x.to_value(u.m), itrs.y.to_value(u.m), itrs.z.to_value(u.m)]).T
    return xyz

model_ecef = teme_to_itrs_xyz(states_model)
sgp4_ecef = teme_to_itrs_xyz(states_sgp4_teme)

fig2d = build_groundtrack_figure(
    trajectories_ecef=[model_ecef, sgp4_ecef],
    names=['Model track', f'SGP4 track: {name}'],
)
fig2d.show()
